# CSE6242 - HW3 - Q1

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove any comment that says "#export" because that will crash the autograder in Gradescope. We use this comment to export your code in these cells for grading.
</div>

Pyspark Imports

In [ ]:
#export
### DO NOT MODIFY THIS CELL ###
import pyspark
from pyspark.sql import SQLContext
from pyspark.sql.functions import hour, when, col, date_format, to_timestamp, ceil, coalesce

Initialize PySpark Context

In [ ]:
### DO NOT MODIFY THIS CELL ###
sc = pyspark.SparkContext(appName="HW3-Q1")
sqlContext = SQLContext(sc)

Define function for loading data

In [ ]:
### DO NOT MODIFY THIS CELL ###
def load_data():
    df = sqlContext.read.option("header",True) \
     .csv("yellow_tripdata_2019-01_short.csv")
    return df

### Q1.1

Perform data casting to clean incoming dataset

In [ ]:
#export

# define how to access the columns and target types
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, FloatType, TimestampType

def clean_data(df):
    '''
    input: df a dataframe
    output: df a dataframe with the all the original columns
    '''
    
    # START YOUR CODE HERE ---------
    vendor_id_name = 'VendorID'
    tpep_pickup_datetime_name = 'tpep_pickup_datetime'
    tpep_dropoff_datetime_name = 'tpep_dropoff_datetime'
    passenger_count_name = 'passenger_count'
    trip_distance_name = 'trip_distance'
    rate_code_id_name = 'RatecodeID'
    store_and_fwd_flag_name = 'store_and_fwd_flag'
    pu_location_id_name = 'PULocationID'
    do_location_id_name = 'DOLocationID'
    payment_type_name = 'payment_type'
    fare_amount_name = 'fare_amount'
    extra_name = 'extra'
    mta_tax_name = 'mta_tax'
    tip_amount_name = 'tip_amount'
    tolls_amount_name = 'tolls_amount'
    improvement_surcharge_name = 'improvement_surcharge'
    total_amount_name = 'total_amount'
    congestion_surcharge_name = 'congestion_surcharge'
    # Incoming columns
    # root
    #  |-- VendorID: string (nullable = true)
    #  |-- tpep_pickup_datetime: string (nullable = true)
    #  |-- tpep_dropoff_datetime: string (nullable = true)
    #  |-- passenger_count: string (nullable = true)
    #  |-- trip_distance: string (nullable = true)
    #  |-- RatecodeID: string (nullable = true)
    #  |-- store_and_fwd_flag: string (nullable = true)
    #  |-- PULocationID: string (nullable = true)
    #  |-- DOLocationID: string (nullable = true)
    #  |-- payment_type: string (nullable = true)
    #  |-- fare_amount: string (nullable = true)
    #  |-- extra: string (nullable = true)
    #  |-- mta_tax: string (nullable = true)
    #  |-- tip_amount: string (nullable = true)
    #  |-- tolls_amount: string (nullable = true)
    #  |-- improvement_surcharge: string (nullable = true)
    #  |-- total_amount: string (nullable = true)
    #  |-- congestion_surcharge: string (nullable = true)

    # Casting columns
    # total_amount — float
    # tip_amount — float
    # trip_distance — float
    # fare_amount — float
    # tpep_pickup_datetime — timestamp
    # tpep_dropoff_datetime — timestamp
    
    df = df.withColumn(passenger_count_name, col(passenger_count_name).cast(IntegerType())) \
            .withColumn(total_amount_name, col(total_amount_name).cast(FloatType())) \
            .withColumn(tip_amount_name, col(tip_amount_name).cast(FloatType())) \
            .withColumn(trip_distance_name, col(trip_distance_name).cast(FloatType())) \
            .withColumn(fare_amount_name, col(fare_amount_name).cast(FloatType())) \
            .withColumn(tpep_pickup_datetime_name, col(tpep_pickup_datetime_name).cast(TimestampType())) \
            .withColumn(tpep_dropoff_datetime_name, col(tpep_dropoff_datetime_name).cast(TimestampType()))

    # END YOUR CODE HERE -----------

    return df

### Q1.2

Find rate per person for based on how many passengers travel between pickup and dropoff locations. 

In [ ]:
#export
from pyspark.sql import functions as F

def common_pair(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - PULocationID
            - DOLocationID
            - total_passenger_count
            - per_person_rate
            
    per_person_rate is the total_amount per person for a given pair.
    
    '''
    
    # START YOUR CODE HERE ---------
    vendor_id_name = 'VendorID'
    tpep_pickup_datetime_name = 'tpep_pickup_datetime'
    tpep_dropoff_datetime_name = 'tpep_dropoff_datetime'
    passenger_count_name = 'passenger_count'
    trip_distance_name = 'trip_distance'
    rate_code_id_name = 'RatecodeID'
    store_and_fwd_flag_name = 'store_and_fwd_flag'
    pu_location_id_name = 'PULocationID'
    do_location_id_name = 'DOLocationID'
    payment_type_name = 'payment_type'
    fare_amount_name = 'fare_amount'
    extra_name = 'extra'
    mta_tax_name = 'mta_tax'
    tip_amount_name = 'tip_amount'
    tolls_amount_name = 'tolls_amount'
    improvement_surcharge_name = 'improvement_surcharge'
    total_amount_name = 'total_amount'
    congestion_surcharge_name = 'congestion_surcharge'
    # You will be modifying the function common_pair. Return the top 10 pickup-dropoff location pairs that have
    # the highest sum of passenger_count who have traveled between them. Sort the location pairs by total passengers
    # between pairs. For each location pair, also compute the average amount per passenger over all trips (name this
    # per_person_rate), utilizing total_amount.
    # For pairs with the same total passengers, sort them in descending order of per_person_rate. Filter out any trips
    # that have the same pick-up and drop-off location.
    # Rename the column for total passengers to total_passenger_count.

    # Incoming
    # root
    #  |-- VendorID: string (nullable = true)
    #  |-- tpep_pickup_datetime: timestamp (nullable = true)
    #  |-- tpep_dropoff_datetime: timestamp (nullable = true)
    #  |-- passenger_count: string (nullable = true)
    #  |-- trip_distance: float (nullable = true)
    #  |-- RatecodeID: string (nullable = true)
    #  |-- store_and_fwd_flag: string (nullable = true)
    #  |-- PULocationID: string (nullable = true)
    #  |-- DOLocationID: string (nullable = true)
    #  |-- payment_type: string (nullable = true)
    #  |-- fare_amount: float (nullable = true)
    #  |-- extra: string (nullable = true)
    #  |-- mta_tax: string (nullable = true)
    #  |-- tip_amount: float (nullable = true)
    #  |-- tolls_amount: string (nullable = true)
    #  |-- improvement_surcharge: string (nullable = true)
    #  |-- total_amount: float (nullable = true)
    #  |-- congestion_surcharge: string (nullable = true)

    # The initial filtering criteria is top 10 pu/do ID locations
    total_passenger_count_name = 'total_passenger_count'
    row_limit = 10
    # And we want the total fare so we can compute an avg on next step
    total_total_amount_name = 'total_total_amount'
    
    df = df \
        .filter(F.col(pu_location_id_name) != F.col(do_location_id_name)) \
        .groupBy(pu_location_id_name, do_location_id_name).agg(
            F.sum(passenger_count_name).alias(total_passenger_count_name),
            F.sum(total_amount_name).alias(total_total_amount_name)
        )
    
    per_person_rate_name = 'per_person_rate'
    df = df.withColumn(per_person_rate_name, df[total_total_amount_name] / df[total_passenger_count_name])

    df = df.orderBy(
        F.desc(total_passenger_count_name),
        F.desc(per_person_rate_name)
    ).limit(row_limit)

    df = df.select(pu_location_id_name, do_location_id_name, total_passenger_count_name, per_person_rate_name)
    # END YOUR CODE HERE -----------
    
    return df

### Q1.3

Find trips which trip distances generate the highest tip percentage.

In [ ]:
#export
from pyspark.sql import functions as F

def distance_with_most_tip(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - trip_distance
            - tip_percent
            
    trip_percent is the percent of tip out of fare_amount
    
    '''
    
    # START YOUR CODE HERE ---------
    vendor_id_name = 'VendorID'
    tpep_pickup_datetime_name = 'tpep_pickup_datetime'
    tpep_dropoff_datetime_name = 'tpep_dropoff_datetime'
    passenger_count_name = 'passenger_count'
    trip_distance_name = 'trip_distance'
    rate_code_id_name = 'RatecodeID'
    store_and_fwd_flag_name = 'store_and_fwd_flag'
    pu_location_id_name = 'PULocationID'
    do_location_id_name = 'DOLocationID'
    payment_type_name = 'payment_type'
    fare_amount_name = 'fare_amount'
    extra_name = 'extra'
    mta_tax_name = 'mta_tax'
    tip_amount_name = 'tip_amount'
    tolls_amount_name = 'tolls_amount'
    improvement_surcharge_name = 'improvement_surcharge'
    total_amount_name = 'total_amount'
    congestion_surcharge_name = 'congestion_surcharge'

    tip_percent_name = 'tip_percent'
    tip_limit = 15
    # Filter the data for trips having fares (fare_amount) greater than $2.00 and a trip distance (trip_distance) greater
    # than 0. Calculate the tip percent (tip_amount * 100 / fare_amount) for each trip. Round all trip distances up
    # to the closest mile and find the average tip_percent for each trip_distance. Sort the result in descending
    # order of tip_percent to obtain the top 15 trip distances which tip the most generously. Rename the column for
    # rounded trip distances to trip_distance, and the column for average tip percents tip_percent.
    df = df.filter((F.col(fare_amount_name) > 2.0) & (F.col(trip_distance_name) > 0.0))
    df = df.withColumn(tip_percent_name, (100.0 * df[tip_amount_name]) / df[fare_amount_name])
    df = df.withColumn(trip_distance_name, F.ceil(df[trip_distance_name]))
    df = df.groupBy(trip_distance_name).agg(F.avg(tip_percent_name).alias(tip_percent_name))
    df = df.orderBy(
        F.desc(tip_percent_name)
    ).limit(tip_limit)
    # END YOUR CODE HERE -----------
    
    return df

### Q1.4

Determine the average speed at different times of day.

In [ ]:
#export
from pyspark.sql import functions as F

def time_with_most_traffic(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - time_of_day
            - am_avg_speed
            - pm_avg_speed
            
    am_avg_speed and pm_avg_speed are the average trip distance / average trip time calculated for each hour
    
    '''
    
    # START YOUR CODE HERE ---------
    vendor_id_name = 'VendorID'
    tpep_pickup_datetime_name = 'tpep_pickup_datetime'
    tpep_dropoff_datetime_name = 'tpep_dropoff_datetime'
    passenger_count_name = 'passenger_count'
    trip_distance_name = 'trip_distance'
    rate_code_id_name = 'RatecodeID'
    store_and_fwd_flag_name = 'store_and_fwd_flag'
    pu_location_id_name = 'PULocationID'
    do_location_id_name = 'DOLocationID'
    payment_type_name = 'payment_type'
    fare_amount_name = 'fare_amount'
    extra_name = 'extra'
    mta_tax_name = 'mta_tax'
    tip_amount_name = 'tip_amount'
    tolls_amount_name = 'tolls_amount'
    improvement_surcharge_name = 'improvement_surcharge'
    total_amount_name = 'total_amount'
    congestion_surcharge_name = 'congestion_surcharge'

    # You will be modifying the function time_with_most_traffic to determine which hour of the day has the
    # most traffic. Calculate the traffic for a particular hour using the average speed of all taxi trips which began during that
    # hour. Calculate the average speed as the average trip_distance divided by the average trip duration, as distance
    # per hour. Make sure to determine the average durations and average trip distances before calculating the speed. It
    # will likely be helpful to cast the dates to the long data type when determining the interval. A day with low average
    # speed indicates high levels of traffic. The average speed may be 0, indicating very high levels of traffic.
    # Additionally, you must separate the hours into AM and PM, with hours 0:00-11:59 being AM, and hours 12:00-23:59
    # being PM. Convert these times to the 12-hour time, so you can match the output below. For example, the row with 1
    # as time of day, should show the average speed between 1 am and 2 am in the am_avg_speed column, and between
    # 1 pm and 2pm in the pm_avg_speed column.
    # Use date_format along with the appropriate pattern letters to format the time of day so that it matches the example
    # output below. Your final table should contain values sorted from 0–11 for time_of_day. There may be data missing
    # for a time of day, and it may be null for am_avg_speed or pm_avg_speed. If an hour has no data for am or pm,
    # there may be missing rows. You will not have rows for all possible times of day, and do not need to add them to the
    # data if they are missing.

    #distance_per
    tpep_pickup_hour_name = 'tpep_pickup_hour'
    trip_duration_hours_name = 'trip_duration_hours'
    df = df \
        .withColumn(tpep_pickup_hour_name, F.hour(F.col(tpep_pickup_datetime_name))) \
        .withColumn(trip_duration_hours_name, (F.col(tpep_dropoff_datetime_name).cast("long") - F.col(tpep_pickup_datetime_name).cast("long")) / 3600.0)
    df = df.groupBy(tpep_pickup_hour_name).agg(
        F.avg(trip_distance_name).alias(trip_distance_name),
        F.avg(trip_duration_hours_name).alias(trip_duration_hours_name)
    )

    distance_per_hour_name = 'distance_per_hour'
    df = df.withColumn(distance_per_hour_name, df[trip_distance_name] / df[trip_duration_hours_name])

    # tpep_pickup_hour|      trip_distance|trip_duration_hours|  distance_per_hour
    am_avg_speed_name = 'am_avg_speed'
    pm_avg_speed_name = 'pm_avg_speed'
    
    df = df.withColumn(am_avg_speed_name,
       F.when(df[tpep_pickup_hour_name] < 12, df[distance_per_hour_name]).otherwise(F.lit(None))
    )
    df = df.withColumn(pm_avg_speed_name,
       F.when(df[tpep_pickup_hour_name] >= 12, df[distance_per_hour_name]).otherwise(F.lit(None))
    )

    time_of_day_name = 'time_of_day'
    df = df.withColumn(time_of_day_name, df[tpep_pickup_hour_name] % 12)

    df = df.groupBy(time_of_day_name).agg(
        F.first(am_avg_speed_name, ignorenulls=True).alias(am_avg_speed_name),
        F.first(pm_avg_speed_name, ignorenulls=True).alias(pm_avg_speed_name)
    ).orderBy(F.asc(time_of_day_name))
    
    # END YOUR CODE HERE -----------
    
    return df

## The below cells are for you to investigate your solutions and will not be graded

In [ ]:
#import IPython

# This will immediately restart the kernel
#IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
df = load_data()
df = clean_data(df)
df.printSchema()

In [ ]:
df = clean_data(load_data())
df.show()

In [ ]:
df = clean_data(load_data())
common_pair(df).show()

In [ ]:
df = clean_data(load_data())
distance_with_most_tip(df).show()

In [ ]:
df = load_data()
df = clean_data(df)
df.printSchema()
time_with_most_traffic(df).show()